Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Like for like: same cafe, same latte

In DS8_1, the median latte rose 41% from 2019 to 2025. But the cafes photographed in 2019 are mostly not the ones photographed in 2025. Part of that 41% could be a change in **which cafes** we see, not in what they charge.

The research team's answer is to compare **like with like**: the same item, at the same venue, over time. That's how official price statistics work too.

The LSE Growth Lab collected **3.3 million photos of menus**, posted with Google Maps reviews across 25 UK cities, and used **large language models** (AI that can read images) to pull out the prices: **12.8 million prices** from 80,000 cafes, pubs and restaurants, from 2011 to 2026. It's part of the AutoCPI project, which uses web data to measure prices in more detail than official statistics can: in June 2026, the ONS collected just 115 hospitality prices for the whole of Wales.

In this notebook:
1. **The simple comparison:** all lattes in 2019 against all lattes in 2025
2. **Same cafe:** matching cafes across years with `merge`
3. **Why they differ:** who drops out
4. **All 12 items:** both measures, in one chart

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()     # Altair stops at 5,000 rows by default: our tables are bigger

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles

<br><br>

---

<br>

## 1. The simple comparison

Read the four columns we need, and add the year:

In [ ]:
items = pd.read_parquet(DATA + "menu_iconic_items.parquet", columns=["venue_id", "canon", "price", "date"])
items["year"] = pd.to_datetime(items["date"]).dt.year

latte = items[items["canon"] == "latte"]
before = latte[latte["year"] == 2019]
after = latte[latte["year"] == 2025]

print(f"2019: median £{before['price'].median():.2f}, from {before['venue_id'].nunique():,} cafes")
print(f"2025: median £{after['price'].median():.2f}, from {after['venue_id'].nunique():,} cafes")
print(f"Change: {after['price'].median() / before['price'].median() - 1:.0%}")

<br>

Up 41%. But the 2019 figure comes from about 1,500 cafes and the 2025 figure from about 7,000. Most of them appear in only one of the two years.

<br><br>

---

<br>

## 2. Same cafe

First, one price per cafe per year. A cafe can have several latte prices (sizes, or several photos), so take each cafe's median:

In [ ]:
cafe_2019 = before.groupby("venue_id")["price"].median().reset_index(name="price_2019")
cafe_2025 = after.groupby("venue_id")["price"].median().reset_index(name="price_2025")
cafe_2019.head()

<br>

Now **merge** the two tables on `venue_id`, the **key** they share. By default, `merge` keeps only the keys found in **both** tables (an *inner join*): cafes with a latte price in 2019 **and** in 2025. Then work out each cafe's change:

In [ ]:
same = cafe_2019.merge(cafe_2025, on="venue_id")
same["change"] = same["price_2025"] / same["price_2019"] - 1

print(f"{len(same):,} cafes in both years")
print(f"Median change at the same cafe: {same['change'].median():.0%}")
same.head()

<br>

Up 34%, not 41%. Both numbers are correct; they answer different questions. 41% is "how much more does a latte cost on today's menus?"; 34% is "how much did cafes raise their own prices?". Inflation measures ask the second question.

### Each cafe's change

`same` has one row per cafe, 369 rows, so we can chart every cafe's change as a histogram. `bin` groups the changes into 5-point steps and `count()` counts the cafes in each. A few changes are huge (one cafe "rose" 31 times: a misread price), so we leave out anything below −50% or above +150%:

In [ ]:
shown = same[same["change"].between(-0.5, 1.5)]

alt.Chart(shown).mark_bar(color="#8c5a2b").encode(
    x=alt.X("change:Q").bin(step=0.05).title("Change in the price of a latte at the same cafe, 2019 to 2025").axis(format=".0%"),
    y=alt.Y("count():Q").title("Cafes"),
    tooltip=[
        alt.Tooltip("change:Q").bin(step=0.05).format(".0%").title("Change"),
        alt.Tooltip("count():Q").title("Cafes"),
    ],
).properties(width=450, height=220,
             title=alt.Title("How much did cafes raise the price of a latte?", subtitle=SOURCE))

<br>

Most cafes raised the price by 20% to 50%. About 1 in 20 cut it.

<br><br>

---

<br>

## 3. Why they differ

Compare the 2019 cafes we see again in 2025 with those we don't. `isin` checks each `venue_id` against the matched list; `~` means "not":

In [ ]:
seen_again = cafe_2019["venue_id"].isin(same["venue_id"])

print(f"2019 cafes also seen in 2025: median £{cafe_2019[seen_again]['price_2019'].median():.2f}")
print(f"2019 cafes not seen in 2025:  median £{cafe_2019[~seen_again]['price_2019'].median():.2f}")

<br>

The cafes that drop out were cheaper. They pull the 2019 median down, and so make the simple change look bigger. This is **composition**: the mix of cafes changed, as well as their prices. (We saw the same thing with Wetherspoon pints in DS8_2.)

<br><br>

---

<br>

## 4. All 12 items

The same steps for every item at once. Group by item, venue **and** year, then `unstack()` the year (as in DS8_3), so each venue has a 2019 column and a 2025 column. `dropna()` keeps only the venues with a price in both:

In [ ]:
two_years = items[items["year"].isin([2019, 2025])]

by_venue = two_years.groupby(["canon", "venue_id", "year"])["price"].median().unstack()
by_venue.columns = ["price_2019", "price_2025"]
by_venue = by_venue.dropna()                         # venues with the item in both years
by_venue["change"] = by_venue["price_2025"] / by_venue["price_2019"] - 1
by_venue.head()

<br>

Now the two measures for each item. The same-venue change is the median of the venues' changes; the simple change compares the medians of all prices. `join` lines up two tables by their index (here, the item):

In [ ]:
same_venue = by_venue.groupby("canon")["change"].agg(["median", "count"])
same_venue.columns = ["same_venue", "venues"]

medians = two_years.groupby(["canon", "year"])["price"].median().unstack()
medians["all_menus"] = medians[2025] / medians[2019] - 1

compare = same_venue.join(medians["all_menus"]).reset_index()
compare = compare[compare["venues"] >= 50]           # leave out items matched at few venues
compare.round(3)

<br>

A **dumbbell chart**: a grey dot for all menus, a red dot for the same venue, and a line between them. Three layers, sharing the y-axis:

In [ ]:
y = alt.Y("canon:N").sort(field="all_menus", order="descending").title(None)

line = alt.Chart(compare).mark_rule(color="#bbbbbb", strokeWidth=3).encode(
    y=y,
    x=alt.X("all_menus:Q").title("Change in price, 2019 to 2025").axis(format=".0%"),
    x2="same_venue:Q",
)
all_menus = alt.Chart(compare).mark_circle(size=110, color="#888888", opacity=1).encode(
    y=y,
    x=alt.X("all_menus:Q"),
    tooltip=[
        alt.Tooltip("canon:N").title("Item"),
        alt.Tooltip("all_menus:Q").format(".0%").title("All menus"),
    ],
)
same_venues = alt.Chart(compare).mark_circle(size=110, color="#c8102e", opacity=1).encode(
    y=y,
    x=alt.X("same_venue:Q"),
    tooltip=[
        alt.Tooltip("canon:N").title("Item"),
        alt.Tooltip("same_venue:Q").format(".0%").title("Same venue"),
        alt.Tooltip("venues:Q").format(",").title("Venues matched"),
    ],
)

chart = (line + all_menus + same_venues).properties(
    width=420, height=260,
    title=alt.Title("Like for like changes the answer",
                    subtitle=["Change in price, 2019 to 2025. Grey: all menus. Red: the same venue.", SOURCE]))
chart

<br>

Matching moves the answer **both ways**. For coffee (latte, cappuccino) and fish & chips, the simple comparison overstates the rise. For margherita and chicken tikka masala, it understates it. A median of whatever was photographed mixes price changes with changes in who was photographed.

Save the chart for your website:

In [ ]:
chart.save("like_for_like.json")

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1. Other years.** Compare 2022 and 2025 instead. Is the gap between the two measures smaller over a shorter period?

**2. Stricter matching.** A cafe's "latte" might be a different size in each photo. Match on the item as written, too: add `"item"` to the columns you read and to the `groupby`. How many matches are left, and does the change move?

**3. Pints.** Use the `pints` table from DS8_2 (read `menu_beer_cider.parquet` and copy the cleaning cell). How much did the same pub raise the price of a pint, 2019 to 2025?

**4. A chart of every cafe.** Make a scatter plot of `price_2019` (x) against `price_2025` (y) for the 369 cafes, with a line where they're equal. How many cafes are below it?

In [ ]:
# Your code here
